# ADMM Fixed-Parameter Run In Colab

This notebook is a thin Colab launcher for fixed-parameter ADMM experiments in this repo.

It will:
- mount Google Drive
- fetch the repo into `/content`
- install minimal Colab-safe dependencies
- download the FFHQ checkpoint if needed
- build a fixed ADMM command using the paper hyperparameters
- launch the run in the background
- show status snapshots and current log tail on demand
- show saved results and copy them back to Drive


## 1. Runtime

In Colab, go to `Runtime > Change runtime type` and choose:
- `Python 3`
- `GPU`

If available, prefer `A100` or better. This notebook is for fixed-parameter runs, not tuning.


In [ ]:
#@title Project Settings

SETUP_MODE = "git"  #@param ["git", "drive_zip"]
REPO_URL = "https://github.com/Seif-Hussein/dyscode.git"  #@param {type:"string"}
REPO_BRANCH = "main"  #@param {type:"string"}
DRIVE_ZIP_PATH = "/content/drive/MyDrive/mycode2.zip"  #@param {type:"string"}

REPO_DIR = "/content/mycode2"
DRIVE_EXPORT_DIR = "/content/drive/MyDrive/admm_runs"  #@param {type:"string"}
DRIVE_FFHQ_DATA_DIR = "/content/drive/MyDrive/mycode/test-ffhq"  #@param {type:"string"}
SESSION_TAG = ""  #@param {type:"string"}

TASK = "phase_retrieval"  #@param ["down_sampling", "gaussian_blur", "hdr", "inpainting_rand", "inpainting", "motion_blur", "nonlinear_blur", "phase_retrieval"]
CONFIG_NAME = "default_ffhq.yaml"  #@param ["default_ffhq.yaml"]

TOTAL_IMAGES = 100  #@param {type:"integer"}
BATCH_SIZE = 10  #@param {type:"integer"}
OPERATOR_SIGMA = 0.05  #@param {type:"number"}
FINAL_STEP = "ode"  #@param ["ode", "tweedie"]
LGVD_NUM_STEPS = 10  #@param {type:"integer"}
SAVE_SAMPLES = True  #@param {type:"boolean"}
SHOW_CONFIG = False  #@param {type:"boolean"}

OVERRIDE_RHO = ""  #@param {type:"string"}
OVERRIDE_W = ""  #@param {type:"string"}
OVERRIDE_ADAM_LR = ""  #@param {type:"string"}

RUN_MANIFEST_PATH = ""


In [ ]:
#@title Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
os.chdir("/content")
print(os.getcwd())
#@title Fetch The Repo
import shutil
import zipfile
from pathlib import Path

repo_dir = Path(REPO_DIR)
if repo_dir.exists():
    shutil.rmtree(repo_dir)

if SETUP_MODE == "git":
    if not REPO_URL:
        raise ValueError("Fill in REPO_URL before running this cell.")
    !git clone --depth 1 --branch "$REPO_BRANCH" "$REPO_URL" "$REPO_DIR"
elif SETUP_MODE == "drive_zip":
    zip_path = Path(DRIVE_ZIP_PATH)
    if not zip_path.exists():
        raise FileNotFoundError(f"Zip file not found: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as handle:
        handle.extractall("/content")
else:
    raise ValueError(f"Unsupported SETUP_MODE: {SETUP_MODE}")

if not repo_dir.exists():
    raise FileNotFoundError(f"Repo directory was not created at {repo_dir}")

%cd /content/mycode2
!git status --short


In [ ]:
#@title Install Python Dependencies
!pip install hydra-core==1.3.2 omegaconf==2.3.0 gdown==5.2.0 piq==0.8.0 prettytable==3.14.0 setproctitle==1.3.3 wandb==0.17.8 imageio==2.37.0


In [ ]:
#@title Download The FFHQ Checkpoint If Needed
%cd /content/mycode2
!mkdir -p pretrained-models
!test -f pretrained-models/ffhq_10m.pt || gdown --id 1BGwhRWUoguF-D8wlZ65tf227gp3cDUDh -O pretrained-models/ffhq_10m.pt
!ls -lh pretrained-models


## 2. Build The Fixed Run

The notebook uses the paper defaults for ADMM by task:
- `rho`
- `W`
- Adam learning rate

For your note about `W`, this notebook sets both:
- `sampler.annealing_scheduler_config.num_steps = W`
- `inverse_task.admm_config.max_iter = W`

It also forces Langevin/DC steps to `10` by default.


In [ ]:
#@title Build The ADMM Run Command
from datetime import datetime
from pathlib import Path
import json
import shlex

PAPER_DEFAULTS = {
    "down_sampling": {"rho": 100, "W": 100, "adam_lr": 3e-2},
    "gaussian_blur": {"rho": 100, "W": 100, "adam_lr": 5e-2},
    "hdr": {"rho": 500, "W": 100, "adam_lr": 3e-2},
    "inpainting_rand": {"rho": 500, "W": 100, "adam_lr": 1e-1},
    "inpainting": {"rho": 500, "W": 100, "adam_lr": 1e-1},
    "motion_blur": {"rho": 100, "W": 100, "adam_lr": 1e-1},
    "nonlinear_blur": {"rho": 300, "W": 400, "adam_lr": 3e-1},
    "phase_retrieval": {"rho": 100, "W": 400, "adam_lr": 1e-1},
}

if TASK not in PAPER_DEFAULTS:
    raise KeyError(f"No paper defaults registered for task: {TASK}")

paper = PAPER_DEFAULTS[TASK]

def optional_float(text, default):
    text = str(text).strip()
    return default if text == "" else float(text)

def optional_int(text, default):
    text = str(text).strip()
    return default if text == "" else int(text)

def sanitize(text: str) -> str:
    out = []
    for ch in text:
        if ch.isalnum() or ch in "-_":
            out.append(ch)
        else:
            out.append("_")
    return "".join(out).strip("_") or "run"

rho = optional_float(OVERRIDE_RHO, paper["rho"])
W = optional_int(OVERRIDE_W, paper["W"])
adam_lr = optional_float(OVERRIDE_ADAM_LR, paper["adam_lr"])

effective_total_images = int(TOTAL_IMAGES)
effective_batch_size = min(int(BATCH_SIZE), effective_total_images)

repo_dir = Path(REPO_DIR)
drive_data_dir = Path(DRIVE_FFHQ_DATA_DIR)
tag = sanitize(SESSION_TAG.strip()) if SESSION_TAG.strip() else ""
suffix = f"_{tag}" if tag else ""
run_timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
run_slug = f"admm_{sanitize(TASK)}{suffix}_{run_timestamp}"
run_name = f"ADMM_{sanitize(TASK)}"

save_root = repo_dir / "results" / "colab_runs" / run_slug
hydra_root = repo_dir / "outputs" / "colab_runs" / run_slug
state_root = repo_dir / "colab_runs"
state_root.mkdir(parents=True, exist_ok=True)
log_path = state_root / f"{run_slug}.log"
pid_path = state_root / f"{run_slug}.pid"
manifest_path = state_root / f"admm_run{suffix or '_default'}.json"
result_dir = save_root / f"{run_name}_ffhq_{TASK}"

cmd = [
    "python",
    "recover_inverse2.py",
    "--config-name",
    CONFIG_NAME,
    "sampler=edm_admm",
    f"inverse_task={TASK}",
    f"name={run_name}",
    "wandb=false",
    f"save_samples={'true' if SAVE_SAMPLES else 'false'}",
    "save_traj=false",
    "save_traj_raw_data=false",
    f"show_config={'true' if SHOW_CONFIG else 'false'}",
    f"total_images={effective_total_images}",
    f"batch_size={effective_batch_size}",
    "num_runs=1",
    f"save_dir={save_root.as_posix()}",
    f"hydra.run.dir={hydra_root.as_posix()}",
    f"inverse_task.operator.sigma={float(OPERATOR_SIGMA)}",
    f"sampler.annealing_scheduler_config.num_steps={W}",
    f"inverse_task.admm_config.max_iter={W}",
    f"inverse_task.admm_config.rho={rho}",
    f"inverse_task.admm_config.ml.lr={adam_lr}",
    f"inverse_task.admm_config.denoise.final_step={FINAL_STEP}",
    f"inverse_task.admm_config.denoise.lgvd.num_steps={int(LGVD_NUM_STEPS)}",
]

if drive_data_dir.exists():
    cmd.append(f"data.image_root_path={drive_data_dir.as_posix()}")
    cmd.append("data.start_idx=0")
    cmd.append("data.end_idx=-1")
else:
    print(f"Drive dataset path not found, keeping repo default data path: {drive_data_dir}")

manifest = {
    "task": TASK,
    "config_name": CONFIG_NAME,
    "paper_defaults": paper,
    "rho": rho,
    "W": W,
    "adam_lr": adam_lr,
    "operator_sigma": float(OPERATOR_SIGMA),
    "final_step": FINAL_STEP,
    "lgvd_num_steps": int(LGVD_NUM_STEPS),
    "total_images": effective_total_images,
    "batch_size": effective_batch_size,
    "run_name": run_name,
    "run_slug": run_slug,
    "save_root": save_root.as_posix(),
    "result_dir": result_dir.as_posix(),
    "grid_results_path": (result_dir / "grid_results.png").as_posix(),
    "metrics_path": (result_dir / "metrics.json").as_posix(),
    "eval_path": (result_dir / "eval.md").as_posix(),
    "log_path": log_path.as_posix(),
    "pid_path": pid_path.as_posix(),
    "command": cmd,
    "built_at": datetime.now().isoformat(timespec="seconds"),
}
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
RUN_MANIFEST_PATH = str(manifest_path)

print(f"Wrote manifest: {manifest_path}")
print(f"Task: {TASK}")
print(f"rho={rho}, W={W}, Adam lr={adam_lr}, final_step={FINAL_STEP}, lgvd.num_steps={int(LGVD_NUM_STEPS)}")
print(f"Result dir: {result_dir}")
print("
Command:")
print(" ".join(shlex.quote(part) for part in cmd))


In [ ]:
#@title Dry Run The ADMM Command
from pathlib import Path
import json
import shlex

manifest_path = Path(RUN_MANIFEST_PATH)
if not manifest_path.exists():
    raise FileNotFoundError(f"Run manifest not found: {manifest_path}")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
print(json.dumps({k: v for k, v in manifest.items() if k != 'command'}, indent=2))
print("
Command:")
print(" ".join(shlex.quote(part) for part in manifest["command"]))


In [ ]:
#@title Launch ADMM In Background
from pathlib import Path
from datetime import datetime
import json
import subprocess

manifest_path = Path(RUN_MANIFEST_PATH)
if not manifest_path.exists():
    raise FileNotFoundError(f"Run manifest not found: {manifest_path}")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

log_path = Path(manifest["log_path"])
pid_path = Path(manifest["pid_path"])
save_root = Path(manifest["save_root"])
result_dir = Path(manifest["result_dir"])
log_path.parent.mkdir(parents=True, exist_ok=True)
save_root.mkdir(parents=True, exist_ok=True)
result_dir.parent.mkdir(parents=True, exist_ok=True)

with log_path.open("w", encoding="utf-8") as handle:
    proc = subprocess.Popen(
        manifest["command"],
        cwd=Path(REPO_DIR),
        stdout=handle,
        stderr=subprocess.STDOUT,
        text=True,
    )

pid_path.write_text(str(proc.pid), encoding="utf-8")
manifest["pid"] = proc.pid
manifest["started_at"] = datetime.now().isoformat(timespec="seconds")
manifest["status"] = "running"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(f"Launched PID {proc.pid}")
print(f"Log: {log_path}")
print(f"Result dir: {result_dir}")


In [ ]:
#@title Refresh Status Snapshot
from pathlib import Path
import json
import os
from IPython.display import Image, display

manifest_path = Path(RUN_MANIFEST_PATH)
if not manifest_path.exists():
    raise FileNotFoundError(f"Run manifest not found: {manifest_path}")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

pid = manifest.get("pid")
alive = False
if pid is not None:
    try:
        os.kill(int(pid), 0)
        alive = True
    except OSError:
        alive = False

result_dir = Path(manifest["result_dir"])
metrics_path = Path(manifest["metrics_path"])
grid_results_path = Path(manifest["grid_results_path"])
eval_path = Path(manifest["eval_path"])

if metrics_path.exists():
    status = "completed"
elif alive:
    status = "running"
else:
    status = "stopped_or_failed"

print(f"Status: {status}")
print(f"Task: {manifest['task']}")
print(f"Built at: {manifest['built_at']}")
print(f"Started at: {manifest.get('started_at', 'n/a')}")
print(f"PID: {manifest.get('pid', 'n/a')} (alive={alive})")
print(f"Result dir: {result_dir}")
print(f"Log path: {manifest['log_path']}")
print(f"rho={manifest['rho']} W={manifest['W']} Adam lr={manifest['adam_lr']} final_step={manifest['final_step']} lgvd.num_steps={manifest['lgvd_num_steps']}")
print(f"total_images={manifest['total_images']} batch_size={manifest['batch_size']}")

if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    print()
    print("Metrics summary:")
    for key in ["psnr", "ssim", "lpips"]:
        if key in metrics:
            cmp_key = "max" if key in {"psnr", "ssim"} else "min"
            vals = metrics[key][cmp_key]
            mean_val = sum(vals) / len(vals)
            print(f"  {key}: mean {cmp_key} = {mean_val:.4f}")

if grid_results_path.exists():
    print()
    print(f"Grid results: {grid_results_path}")
    display(Image(filename=str(grid_results_path)))

if eval_path.exists():
    print()
    print("Eval markdown:")
    print(eval_path.read_text(encoding="utf-8", errors="replace"))


In [ ]:
#@title Refresh Current Log Tail
from pathlib import Path
import json

manifest_path = Path(RUN_MANIFEST_PATH)
if not manifest_path.exists():
    raise FileNotFoundError(f"Run manifest not found: {manifest_path}")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
log_path = Path(manifest["log_path"])
print(f"Log file: {log_path}")
if not log_path.exists():
    raise FileNotFoundError(f"Log file not found: {log_path}")

lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
for line in lines[-80:]:
    print(line)


In [ ]:
#@title Show Latest Results
from pathlib import Path
import json
from IPython.display import Image, display

manifest_path = Path(RUN_MANIFEST_PATH)
if not manifest_path.exists():
    raise FileNotFoundError(f"Run manifest not found: {manifest_path}")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
result_dir = Path(manifest["result_dir"])
metrics_path = Path(manifest["metrics_path"])
grid_results_path = Path(manifest["grid_results_path"])
eval_path = Path(manifest["eval_path"])

print(f"Result dir: {result_dir}")
if not result_dir.exists():
    raise FileNotFoundError(f"Result dir not found: {result_dir}")

print("Files:")
for path in sorted(result_dir.iterdir()):
    print(f"- {path.name}")

if grid_results_path.exists():
    print()
    print(f"Grid results: {grid_results_path}")
    display(Image(filename=str(grid_results_path)))

if eval_path.exists():
    print()
    print("Eval markdown:")
    print(eval_path.read_text(encoding="utf-8", errors="replace"))

if metrics_path.exists():
    print()
    print("Metrics JSON:")
    print(json.dumps(json.loads(metrics_path.read_text(encoding="utf-8")), indent=2))


In [ ]:
#@title Copy Latest Results Back To Google Drive
from pathlib import Path
import json
import shutil

manifest_path = Path(RUN_MANIFEST_PATH)
if not manifest_path.exists():
    raise FileNotFoundError(f"Run manifest not found: {manifest_path}")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
save_root = Path(manifest["save_root"])
if not save_root.exists():
    raise FileNotFoundError(f"Save root not found: {save_root}")

export_root = Path(DRIVE_EXPORT_DIR)
export_root.mkdir(parents=True, exist_ok=True)
target = export_root / save_root.name
if target.exists():
    shutil.rmtree(target)
shutil.copytree(save_root, target)
print(f"Copied results to {target}")


In [ ]:
#@title Stop Background Run
from pathlib import Path
import json
import os
import signal

manifest_path = Path(RUN_MANIFEST_PATH)
if not manifest_path.exists():
    raise FileNotFoundError(f"Run manifest not found: {manifest_path}")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
pid = manifest.get("pid")
if pid is None:
    raise ValueError("No PID recorded in manifest.")

os.kill(int(pid), signal.SIGTERM)
print(f"Sent SIGTERM to PID {pid}")


## 3. Next Iteration

Typical workflow:
1. run setup cells
2. build the command
3. dry-run it once
4. launch in background
5. use the snapshot cells to inspect status and logs
6. export results back to Drive when finished
